In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph
from typing import TypedDict
import os 
from dotenv import load_dotenv
load_dotenv()

# Load txt
loader = TextLoader("documents/data.txt")
docs = loader.load()

# Split text
splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
documents = splitter.split_documents(docs)

# Embeddings and 
embeddings = HuggingFaceEmbeddings()

# Vector DB
vectorstore = FAISS.from_documents(documents, embeddings)

retriever = vectorstore.as_retriever()

# LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

# State
class State(TypedDict):
    question: str
    context: str
    answer: str


# Retriever Agent
def retriever_agent(state):

    docs = retriever.invoke(state["question"])

    if docs:
      
        context = "\n".join([doc.page_content for doc in docs])
    else:
        context = "No relevant information found."
    return {
        "question": state["question"],
        "context": context
}

# Answer Agent
from langchain_core.messages import HumanMessage
def answer_agent(state):

    prompt = f"""
    Answer the question using this context.

    Context:
    {state['context']}

    Question:
    {state['question']}
    """

    response = llm.invoke([HumanMessage(content=prompt)])

    return {"answer": response.content}


# Graph
builder = StateGraph(State)

builder.add_node("retriever", retriever_agent)
builder.add_node("answer", answer_agent)

builder.set_entry_point("retriever")

builder.add_edge("retriever", "answer")

builder.set_finish_point("answer")


graph = builder.compile()

# Run
question = input("Ask: ")


result = graph.invoke({"question": question})

print("\nAI",result["answer"])

c:\Users\rushitechnologies\Desktop\mmm\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rushitechnologies\AppData\Local\Temp\ipykernel_13132\1229426567.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
C:\Users\rushitechnologies\AppData\Local\Temp\ipykernel_13132\1229426567.py:22: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_nam


AI Based on the context, it seems like you meant to ask about "LangGraph" rather than "Langggtaph". Here's the answer:

**What is LangGraph?**
LangGraph is a component of the LangChain framework that is used to create stateful multi-agent workflows. This means it is designed to allow developers to build complex applications that involve multiple agents (e.g. LLMs, databases, APIs) interacting with each other in a stateful way, i.e., their interactions depend on the state of the system.

**Five bullet points about LangGraph:**

* **Stateful workflows**: LangGraph allows developers to create complex workflows that involve multiple agents interacting with each other in a stateful way.
* **Multi-agent support**: LangGraph is designed to support multiple agents, such as LLMs, databases, and APIs, to interact with each other in a seamless way.
* **Integration with LangChain**: LangGraph is a part of the LangChain framework, which means it can be easily integrated with other components of th